<a href="https://colab.research.google.com/github/juanfebotero/juanfebotero.github.io/blob/main/notebooks/seguridad-informacion/01_criptografia_practica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Criptografía práctica: de César a AES-GCM y RSA

**Seguridad de la Información** · Prof. Juan Felipe Botero · Universidad de Antioquia

**Objetivos**

- Romper un cifrado clásico por fuerza bruta y por análisis de frecuencias.
- Observar el efecto avalancha de una función hash.
- Cifrar y autenticar con AES-GCM y detectar manipulaciones.
- Construir RSA con números pequeños para entender sus fundamentos.

> **Antes de empezar:** use *Archivo → Guardar una copia en Drive*. Si el cuaderno le llegó por Google Classroom, ya es su copia personal y el profesor verá su progreso.

In [ ]:
!pip -q install cryptography
import hashlib, os, collections, string
import matplotlib.pyplot as plt

## 1. Cifrado César
Cada letra se desplaza $k$ posiciones: $E_k(x) = (x + k) \bmod 26$.

In [ ]:
def cesar(texto, k):
    return "".join(chr((ord(c) - 97 + k) % 26 + 97) if c.isalpha() else c for c in texto.lower())
m = "ataque al amanecer"
c = cesar(m, 3); print(c, "->", cesar(c, -3))

## 2. Criptoanálisis
Solo hay 26 claves: la fuerza bruta es inmediata. El **análisis de frecuencias** es más elegante: en español la letra más frecuente suele ser la *e* (o la *a*).

In [ ]:
cifrado = 'sh zlnbypkhk kl buh ylk klwlukl ahuav kl sh jypwavnyhmph jvtv kl shz wlyzvuhz xbl sh hktpupzayhu'

frec = collections.Counter(ch for ch in cifrado if ch.isalpha())
letras = string.ascii_lowercase
plt.bar(list(letras), [frec.get(l, 0) for l in letras]); plt.title("Frecuencias en el texto cifrado"); plt.show()
print("Más frecuentes:", frec.most_common(3))

**Ejercicio 1.** Encuentre la clave $k$ con la que se cifró `cifrado` y guárdela en `resp_1` (un entero entre 0 y 25). Muestre el texto descifrado.

In [ ]:
resp_1 = None

## 3. Funciones hash y efecto avalancha
Un cambio mínimo en la entrada cambia aproximadamente la mitad de los bits de la salida.

In [ ]:
def bits_distintos(a, b):
    ha, hb = hashlib.sha256(a.encode()).digest(), hashlib.sha256(b.encode()).digest()
    return sum(bin(x ^ y).count("1") for x, y in zip(ha, hb))
print(hashlib.sha256(b"Transferir 100 COP").hexdigest())
print(hashlib.sha256(b"Transferir 900 COP").hexdigest())
print("Bits distintos:", bits_distintos("Transferir 100 COP", "Transferir 900 COP"), "de 256")

## 4. AES-GCM: confidencialidad + integridad
GCM es un modo de cifrado **autenticado**: si alguien altera un solo byte del texto cifrado, el descifrado falla.

In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
clave = AESGCM.generate_key(bit_length=256)
aes = AESGCM(clave); nonce = os.urandom(12)
ct = aes.encrypt(nonce, b"Lectura del medidor: 1532 kWh", b"subestacion-7")
print(ct.hex())
print(aes.decrypt(nonce, ct, b"subestacion-7"))

**Ejercicio 2.** Modifique un byte de `ct` (por ejemplo con `bytearray`) e intente descifrar dentro de un `try/except`. Guarde en `resp_2` `True` si el descifrado **falló** como se esperaba.

In [ ]:
resp_2 = None

## 5. RSA de juguete
1. Elegir primos $p, q$; $n = pq$, $\varphi(n) = (p-1)(q-1)$.
2. Elegir $e$ coprimo con $\varphi(n)$.
3. $d = e^{-1} \bmod \varphi(n)$.
4. Cifrar: $c = m^e \bmod n$. Descifrar: $m = c^d \bmod n$.

In [ ]:
p, q, e = 61, 53, 17
n, phi = p*q, (p-1)*(q-1)
# Ejercicio 3: calcule d (pista: pow(e, -1, phi))
d = None
resp_3 = d
if d:
    m = 65; c = pow(m, e, n); print("cifrado:", c, "descifrado:", pow(c, d, n))

In [ ]:
# --- Autoevaluación: no modifique esta celda ---
import math
def verificar(nombre, valor, esperado, tol=1e-6):
    """Compara su respuesta con la esperada y muestra el resultado."""
    if valor is None:
        print(f"[ ] {nombre}: pendiente, complete la celda anterior.")
        return False
    try:
        ok = abs(float(valor) - float(esperado)) <= tol * max(1, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = valor == esperado
    print(f"[{'✔' if ok else '✘'}] {nombre}: {'correcto' if ok else 'revise su respuesta (obtuvo ' + repr(valor) + ')'}")
    return ok

verificar('Ejercicio 1', resp_1, 7)
verificar('Ejercicio 2', resp_2, True)
verificar('Ejercicio 3', resp_3, 2753)

## Para discutir
- ¿Por qué nunca se debe reutilizar un *nonce* con la misma clave en AES-GCM?
- ¿Por qué RSA con primos de 2 dígitos es inseguro? Intente factorizar $n$ por fuerza bruta.
- Abra un sitio web en su navegador y revise su certificado TLS: ¿qué algoritmos usa?